# CRISP-DM Phase 4: Hyperparameter Tuning — All Configurations
## GridSearchCV — RF and SVM across both feature tracks
This notebook tunes all four viable model configurations:
  - Random Forest: All Features
  - Random Forest: Selected Features (k=7)
  - SVM: All Features
  - SVM: Selected Features (k=7)

Method: GridSearchCV with 5-fold stratified cross-validation, scoring='f1'.
np.arange() is used for float parameter ranges because Python's range()
only accepts integers and cannot produce decimal step sequences.
All four results are compared at the end to identify the champion.

In [16]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report, f1_score

# ── Build all-features dataset ─────────────────────────────────────────────
df = joblib.load('../data/cleaned_data.pkl')

# Target column contains strings 'yes'/'no' — map to binary integers
df['personal_loan'] = df['personal_loan'].map({'yes': 1, 'no': 0})

X = df.drop('personal_loan', axis=1)
y = df['personal_loan']

X_train_all, X_test_all, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train_all = X_train_all.copy()
X_test_all  = X_test_all.copy()

# One-hot encode per column
categorical_cols = ['education_level', 'credit_card_acct', 'online_acct']

for col in categorical_cols:
    dummies_train = pd.get_dummies(X_train_all[col], prefix=col)
    X_train_all.drop([col], axis=1, inplace=True)
    X_train_all = pd.concat([X_train_all, dummies_train], axis=1)

    dummies_test = pd.get_dummies(X_test_all[col], prefix=col)
    X_test_all.drop([col], axis=1, inplace=True)
    X_test_all = pd.concat([X_test_all, dummies_test], axis=1)

X_test_all = X_test_all.reindex(columns=X_train_all.columns, fill_value=0)

# Standardise per column
continuous_cols = ['age', 'yrs_experience', 'family_size',
                   'income', 'mortgage_amt', 'credit_card_spend']

scaler = StandardScaler()
for col in continuous_cols:
    scaler.fit(X_train_all[col].values.reshape(-1, 1))
    X_train_all[col] = scaler.transform(X_train_all[col].values.reshape(-1, 1))
    X_test_all[col]  = scaler.transform(X_test_all[col].values.reshape(-1, 1))

print(f"All-features — Train: {X_train_all.shape} | Test: {X_test_all.shape}")

# ── Load selected-features dataset from feature-selection.ipynb ────────────
X_train_sel = joblib.load('../data/X_train_sel.pkl').copy()
X_test_sel  = joblib.load('../data/X_test_sel.pkl').copy()

# Standardise selected features
sel_continuous = [col for col in X_train_sel.columns
                  if col in continuous_cols]

for col in sel_continuous:
    scaler.fit(X_train_sel[col].values.reshape(-1, 1))
    X_train_sel[col] = scaler.transform(X_train_sel[col].values.reshape(-1, 1))
    X_test_sel[col]  = scaler.transform(X_test_sel[col].values.reshape(-1, 1))

print(f"Selected-features — Train: {X_train_sel.shape} | Test: {X_test_sel.shape}")

All-features — Train: (4800, 15) | Test: (1200, 15)
Selected-features — Train: (4800, 7) | Test: (1200, 7)


## Tuning 1: Random Forest — All Features

In [17]:
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth':    [None, 10, 20],
    'criterion':    ['gini', 'entropy']
}

rf_grid_all = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_param_grid,
    verbose=0,
    cv=5,
    scoring='f1',
    n_jobs=-1
)
rf_grid_all.fit(X_train_all, y_train)

print("=== RF GridSearchCV — All Features ===")
print(f"Best params:  {rf_grid_all.best_params_}")
print(f"Best CV F1:   {rf_grid_all.best_score_:.4f}")

=== RF GridSearchCV — All Features ===
Best params:  {'criterion': 'gini', 'max_depth': 10, 'n_estimators': 200}
Best CV F1:   0.7257


## Tuning 2: Random Forest — Selected Features (k=7)

In [18]:
rf_grid_sel = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_param_grid,
    verbose=0,
    cv=5,
    scoring='f1',
    n_jobs=-1
)
rf_grid_sel.fit(X_train_sel, y_train)

print("=== RF GridSearchCV — Selected Features (k=7) ===")
print(f"Best params:  {rf_grid_sel.best_params_}")
print(f"Best CV F1:   {rf_grid_sel.best_score_:.4f}")

=== RF GridSearchCV — Selected Features (k=7) ===
Best params:  {'criterion': 'gini', 'max_depth': 10, 'n_estimators': 100}
Best CV F1:   0.7133


## Tuning 3: SVM — All Features
np.arange() generates float C values because Python's range()
only accepts integers and cannot produce decimal step sequences.

In [19]:
svm_param_grid = {
    'C':            np.arange(0.5, 2.5, 0.5),
    'kernel':       ['linear', 'rbf'],
    'class_weight': [None, 'balanced']
}

svm_grid_all = GridSearchCV(
    SVC(random_state=42),
    svm_param_grid,
    verbose=0,
    cv=5,
    scoring='f1',
    n_jobs=-1
)
svm_grid_all.fit(X_train_all, y_train)

print("=== SVM GridSearchCV — All Features ===")
print(f"Best params:  {svm_grid_all.best_params_}")
print(f"Best CV F1:   {svm_grid_all.best_score_:.4f}")

=== SVM GridSearchCV — All Features ===
Best params:  {'C': np.float64(0.5), 'class_weight': None, 'kernel': 'linear'}
Best CV F1:   0.7119


## Tuning 4: SVM — Selected Features (k=7)
NOTE: The baseline for this configuration achieved F1 near zero, indicating
catastrophic failure. Tuning is attempted here for methodological completeness
but this configuration is not expected to be viable for deployment.

In [20]:
svm_grid_sel = GridSearchCV(
    SVC(random_state=42),
    svm_param_grid,
    verbose=0,
    cv=5,
    scoring='f1',
    n_jobs=-1
)
svm_grid_sel.fit(X_train_sel, y_train)

print("=== SVM GridSearchCV — Selected Features (k=7) ===")
print(f"Best params:  {svm_grid_sel.best_params_}")
print(f"Best CV F1:   {svm_grid_sel.best_score_:.4f}")

=== SVM GridSearchCV — Selected Features (k=7) ===
Best params:  {'C': np.float64(2.0), 'class_weight': None, 'kernel': 'rbf'}
Best CV F1:   0.6988


## Results Summary — Champion Selection
All four tuned configurations ranked by CV F1.
Highest score = champion, carried forward to 07-champion-evaluation.ipynb.

In [21]:
results = {
    'RF — All Features':    rf_grid_all.best_score_,
    'RF — Selected (k=7)':  rf_grid_sel.best_score_,
    'SVM — All Features':   svm_grid_all.best_score_,
    'SVM — Selected (k=7)': svm_grid_sel.best_score_,
}

print("=" * 50)
print("TUNING RESULTS — CV F1 RANKING")
print("=" * 50)
for i, (name, score) in enumerate(
        sorted(results.items(), key=lambda x: x[1], reverse=True), 1):
    marker = "  <-- CHAMPION" if i == 1 else ""
    print(f"{i}. {name}: {score:.4f}{marker}")

print("=" * 50)
champion_name = max(results, key=results.get)
print(f"\nChampion: {champion_name}")
if 'All Features' in champion_name and 'RF' in champion_name:
    print(f"Best params: {rf_grid_all.best_params_}")
elif 'Selected' in champion_name and 'RF' in champion_name:
    print(f"Best params: {rf_grid_sel.best_params_}")
elif 'All Features' in champion_name and 'SVM' in champion_name:
    print(f"Best params: {svm_grid_all.best_params_}")
else:
    print(f"Best params: {svm_grid_sel.best_params_}")

TUNING RESULTS — CV F1 RANKING
1. RF — All Features: 0.7257  <-- CHAMPION
2. RF — Selected (k=7): 0.7133
3. SVM — All Features: 0.7119
4. SVM — Selected (k=7): 0.6988

Champion: RF — All Features
Best params: {'criterion': 'gini', 'max_depth': 10, 'n_estimators': 200}
